# Matemáticas de la Inteligencia Artificial
## Sesión 10 — Lenguaje como problema probabilístico: tokens, cadenas y bigramas

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CuentosCuanticos/matematicas-ia/blob/main/10_bigrama/laboratorio.ipynb)

### Pregunta de la sesión
**¿Cuál es el modelo de lenguaje más pequeño que podemos construir y qué significa matemáticamente predecir el siguiente símbolo?**

Recorrido: $\text{texto}\to\text{tokens}\to\text{índices}\to N\to P\to\text{muestreo autorregresivo}$.

Completa los bloques `TODO`. El objetivo no es usar una biblioteca de NLP, sino construir explícitamente el modelo de caracteres con Python y NumPy.

## Diccionario matemática–código

| Matemática | Código | Significado |
|---|---|---|
| $\mathcal V$ | `vocabulario` | conjunto de tokens |
| $\iota(t)$ | `token_a_id[t]` | índice del token |
| $N_{ij}$ | `N[i,j]` | conteos $v_i\to v_j$ |
| $P_{ij}$ | `P[i,j]` | $p(v_j\mid v_i)$ |
| $\langle B\rangle,\langle E\rangle$ | `BOS`, `EOS` | inicio y final |

Recuerda: un índice es una etiqueta discreta, no una coordenada semántica.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

np.set_printoptions(precision=4, suppress=True)
rng=np.random.default_rng(10)

# 1. Tokenizar y codificar

Trabajaremos con caracteres y con los tokens especiales `BOS` y `EOS`. Para una palabra $t_1\ldots t_n$ queremos representar $\langle B\rangle,t_1,\ldots,t_n,\langle E\rangle$ mediante índices enteros.

In [ ]:
corpus=['casa','cama','capa','masa','mapa','sala','mala']
BOS='<BOS>'; EOS='<EOS>'

# TODO 1: caracteres distintos, vocabulario y diccionarios.
caracteres=...
vocabulario=[BOS]+caracteres+[EOS]
V=len(vocabulario)
token_a_id=...
id_a_token=...

def codificar(palabra):
    tokens=[BOS]+list(palabra)+[EOS]
    # TODO 2
    return ...

def decodificar(ids):
    # TODO 3
    tokens=...
    return ''.join(t for t in tokens if t not in (BOS,EOS))

ids=codificar('casa')
print(vocabulario)
print('casa ->',ids,'->',decodificar(ids))

# 2. El texto crea sus propios ejemplos de siguiente token

Cada pareja $(t_i,t_{i+1})$ puede leerse como $\text{contexto}\to\text{siguiente token}$. Esta es la versión mínima de aprendizaje autosupervisado que necesitamos aquí.

In [ ]:
def pares_bigrama(palabra):
    ids=codificar(palabra)
    # TODO 4
    return ...

print([(id_a_token[i],id_a_token[j]) for i,j in pares_bigrama('casa')])

# Línea base unigrama: p(t)=N(t)/N.
texto=''.join(corpus); cu=Counter(texto)
freq=np.array([cu[c] for c in caracteres],dtype=float)
p_uni=... # TODO 5
print('p_uni=',p_uni,'suma=',p_uni.sum())
print('muestra unigrama=', ''.join(rng.choice(caracteres,size=40,p=...))) # TODO 6

# 3. Aprender contando: $N$ y $P$

Definimos $N_{ij}=\#\{v_i\to v_j\}$ y estimamos por máxima verosimilitud

$$P_{ij}=\widehat p(v_j\mid v_i)=\frac{N_{ij}}{\sum_k N_{ik}}.$$

Esta fórmula no es una heurística: maximiza $\ell_i=\sum_jN_{ij}\log p_{ij}$ bajo $\sum_jp_{ij}=1$.

In [ ]:
def matriz_conteos(corpus,V):
    N=np.zeros((V,V),dtype=int)
    for palabra in corpus:
        for i,j in pares_bigrama(palabra):
            # TODO 7
            ...
    return N

def normalizar_filas(N):
    tot=N.sum(axis=1,keepdims=True)
    P=np.zeros_like(N,dtype=float)
    # TODO 8: usa np.divide(..., where=tot!=0).
    ...
    return P

N=matriz_conteos(corpus,V); P=normalizar_filas(N)
print('N=\n',N); print('P=\n',P)
activas=N.sum(axis=1)>0
print('error normalización=',...) # TODO 9

plt.figure(figsize=(7,6)); plt.imshow(N)
plt.xticks(range(V),vocabulario,rotation=90); plt.yticks(range(V),vocabulario)
plt.xlabel('siguiente token'); plt.ylabel('token actual'); plt.colorbar(label='conteo'); plt.tight_layout(); plt.show()

# 4. Verificación de máxima verosimilitud

Para el contexto `a`, compara la distribución MLE con una distribución uniforme sobre los sucesores observados.

In [ ]:
def loglikelihood_fila(conteos,probs):
    m=conteos>0
    return np.sum(conteos[m]*np.log(probs[m]))

id_a=token_a_id['a']; conteos_a=N[id_a].astype(float); p_mle=P[id_a]
obs=conteos_a>0; p_unif=np.zeros(V); p_unif[obs]=1/obs.sum()
ll_mle=... # TODO 10
ll_unif=... # TODO 11
print('ll MLE=',ll_mle,'ll uniforme=',ll_unif,'MLE>=uniforme?',ll_mle>=ll_unif)

# 5. Generación autorregresiva y probabilidad de secuencias

El bigrama usa $p(t_i\mid t_{<i})\approx p(t_i\mid t_{i-1})$. Para generar, muestreamos $t_{n+1}\sim P_{t_n,:}$ y repetimos hasta `EOS`. Para evaluar una secuencia multiplicamos sus transiciones.

In [ ]:
def generar(P,max_pasos=30,greedy=False):
    actual=token_a_id[BOS]; out=[]
    for _ in range(max_pasos):
        probs=P[actual]
        if probs.sum()==0: break
        # TODO 12: argmax si greedy; si no, rng.choice con p=probs.
        sig=...
        tok=id_a_token[int(sig)]
        if tok==EOS: break
        out.append(tok); actual=int(sig)
    return ''.join(out)

def probabilidad_bigrama(palabra,P):
    ids=codificar(palabra); p=1.0
    for i,j in zip(ids[:-1],ids[1:]):
        # TODO 13
        p*=...
    return p

for _ in range(20): print(generar(P))
for s in ['casa','cama','casama','sapa']:
    print(s,probabilidad_bigrama(s,P))

# 6. Conteos cero y suavizado

MLE asigna probabilidad cero a toda transición no observada. Como experimento opcional:

$$\widetilde P_{ij}=\frac{N_{ij}+\alpha}{N_i+\alpha V}.$$

In [ ]:
def suavizar(N,alpha=1.0):
    V=N.shape[1]
    # TODO 14
    return ...

P1=suavizar(N,1.0)
id_c=token_a_id['c']; id_s=token_a_id['s']
print('sin suavizado p(s|c)=',P[id_c,id_s])
print('con suavizado p(s|c)=',P1[id_c,id_s])

# Problema final abierto — Construir un microgenerador científico de caracteres

No hay una única solución correcta. La versión docente contiene una solución de referencia.

Usa el corpus siguiente y entrega código, resultados y una conclusión razonada.

### Requisitos
1. Decide y justifica el preprocesamiento; conserva o elimina espacios solo de forma razonada.
2. Construye vocabulario y prueba reversibilidad codificar–decodificar.
3. Construye un unigrama y genera al menos 100 caracteres.
4. Construye $N$ y $P$; verifica no negatividad y normalización.
5. Para al menos cinco tokens, incluido el espacio, muestra sus cinco sucesores principales.
6. Genera al menos 30 secuencias y analiza cinco.
7. Evalúa tres secuencias: una observada, una nueva con transiciones conocidas y una con alguna transición no observada.
8. Prueba $\alpha\in\{0.1,0.5,1\}$ y discute el compromiso del suavizado.
9. Compara muestreo y `argmax`.
10. Explica cómo construirías un trigrama y por qué aparece escasez de datos.
11. Concluye en 10–15 líneas: ¿qué ha aprendido el bigrama y qué le falta para representar contexto de largo alcance?

In [ ]:
corpus_reto=[
 'la luz se propaga',
 'la luz tiene energia',
 'la materia tiene masa',
 'la masa curva el espacio',
 'la energia curva el espacio',
 'el espacio tiene geometria',
 'una maquina aprende patrones',
 'un modelo aprende del texto',
 'un modelo predice tokens',
 'el lenguaje es una secuencia',
 'la probabilidad modela incertidumbre',
 'un token sigue a otro token',
]

# RETO A — preprocesamiento y representación
...
# RETO B — unigrama, N y P
...
# RETO C — diagnóstico, generación y probabilidades
...
# RETO D — smoothing, sampling/argmax y propuesta de trigrama
...